SEC API
 ↓
Get company facts
 ↓
Find revenue concept
 ↓
Filter annual periods
 ↓
Remove repeated periods
 ↓
Determine fiscal year
 ↓
Convert units
 ↓
Calculate growth

CIK = Central Index Key.

It's basically the SEC's unique ID for a company.

In [56]:
import os
import sys

sys.path.append("..")

from src.ingestion.sec_client import SECClient

client = SECClient()

apple = client.get_company_facts("0000320193")

In [57]:
apple.keys()

dict_keys(['cik', 'entityName', 'facts'])

In [58]:
apple["entityName"]

'Apple Inc.'

In [59]:
apple["facts"].keys()

dict_keys(['dei', 'us-gaap'])

In [60]:
us_gaap = apple["facts"]["us-gaap"]

len(us_gaap)

503

In [61]:
list(us_gaap.keys())[:50]

['AccountsPayable',
 'AccountsPayableCurrent',
 'AccountsReceivableNetCurrent',
 'AccruedIncomeTaxesCurrent',
 'AccruedIncomeTaxesNoncurrent',
 'AccruedLiabilities',
 'AccruedLiabilitiesCurrent',
 'AccruedMarketingCostsCurrent',
 'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment',
 'AccumulatedOtherComprehensiveIncomeLossAvailableForSaleSecuritiesAdjustmentNetOfTax',
 'AccumulatedOtherComprehensiveIncomeLossCumulativeChangesInNetGainLossFromCashFlowHedgesEffectNetOfTax',
 'AccumulatedOtherComprehensiveIncomeLossForeignCurrencyTranslationAdjustmentNetOfTax',
 'AccumulatedOtherComprehensiveIncomeLossNetOfTax',
 'AdjustmentsToAdditionalPaidInCapitalSharebasedCompensationRequisiteServicePeriodRecognitionValue',
 'AdjustmentsToAdditionalPaidInCapitalTaxEffectFromShareBasedCompensation',
 'AdvertisingExpense',
 'AllocatedShareBasedCompensationExpense',
 'AllowanceForDoubtfulAccountsReceivableCurrent',
 'AmortizationOfIntangibleAssets',
 'AntidilutiveSecuritiesExcluded

In [62]:
revenue_tags = [
    tag for tag in us_gaap.keys()
    if "Revenue" in tag
]

revenue_tags[:30]

['ContractWithCustomerLiabilityRevenueRecognized',
 'DeferredRevenueCurrent',
 'DeferredRevenueNoncurrent',
 'IncreaseDecreaseInDeferredRevenue',
 'RevenueFromContractWithCustomerExcludingAssessedTax',
 'Revenues',
 'SalesRevenueNet',
 'SalesRevenueServicesGross']

In [63]:
revenue = us_gaap[
    "RevenueFromContractWithCustomerExcludingAssessedTax"
]

In [64]:
revenue.keys()

dict_keys(['label', 'description', 'units'])

In [65]:
print(revenue["label"])

Revenue from Contract with Customer, Excluding Assessed Tax


In [66]:
print(revenue["description"])

Amount, excluding tax collected from customer, of revenue from satisfaction of performance obligation by transferring promised good or service to customer. Tax collected from customer is tax assessed by governmental authority that is both imposed on and concurrent with specific revenue-producing transaction, including, but not limited to, sales, use, value added and excise.


In [67]:
revenue["units"].keys()

dict_keys(['USD'])

In [68]:
revenue_data = revenue["units"]["USD"]
revenue_data[:5]

[{'start': '2016-09-25',
  'end': '2017-09-30',
  'val': 229234000000,
  'accn': '0000320193-19-000119',
  'fy': 2019,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2019-10-31',
  'frame': 'CY2017'},
 {'start': '2017-10-01',
  'end': '2017-12-30',
  'val': 88293000000,
  'accn': '0000320193-19-000010',
  'fy': 2019,
  'fp': 'Q1',
  'form': '10-Q',
  'filed': '2019-01-30'},
 {'start': '2017-10-01',
  'end': '2017-12-30',
  'val': 88293000000,
  'accn': '0000320193-19-000119',
  'fy': 2019,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2019-10-31',
  'frame': 'CY2017Q4'},
 {'start': '2017-10-01',
  'end': '2018-03-31',
  'val': 149430000000,
  'accn': '0000320193-19-000066',
  'fy': 2019,
  'fp': 'Q2',
  'form': '10-Q',
  'filed': '2019-05-01'},
 {'start': '2017-12-31',
  'end': '2018-03-31',
  'val': 61137000000,
  'accn': '0000320193-19-000066',
  'fy': 2019,
  'fp': 'Q2',
  'form': '10-Q',
  'filed': '2019-05-01'}]

In [69]:
import pandas as pd

revenue_df = pd.DataFrame(revenue_data)

revenue_df.head()

,start,end,val,accn,fy,fp,form,filed,frame
0,2016-09-25,2017-09-30,229234000000,0000320193-19-000119,2019,FY,10-K,2019-10-31,CY2017
1,2017-10-01,2017-12-30,88293000000,0000320193-19-000010,2019,Q1,10-Q,2019-01-30,NaN
2,2017-10-01,2017-12-30,88293000000,0000320193-19-000119,2019,FY,10-K,2019-10-31,CY2017Q4
3,2017-10-01,2018-03-31,149430000000,0000320193-19-000066,2019,Q2,10-Q,2019-05-01,NaN
4,2017-12-31,2018-03-31,61137000000,0000320193-19-000066,2019,Q2,10-Q,2019-05-01,NaN


In [70]:
revenue_df["form"].value_counts()

form
10-Q    80
10-K    37
Name: count, dtype: int64

In [71]:
annual_revenue = revenue_df[
    revenue_df["form"] == "10-K"
].copy()
annual_revenue.head()

,start,end,val,accn,fy,fp,form,filed,frame
0,2016-09-25,2017-09-30,229234000000,0000320193-19-000119,2019,FY,10-K,2019-10-31,CY2017
2,2017-10-01,2017-12-30,88293000000,0000320193-19-000119,2019,FY,10-K,2019-10-31,CY2017Q4
5,2017-12-31,2018-03-31,61137000000,0000320193-19-000119,2019,FY,10-K,2019-10-31,CY2018Q1
8,2018-04-01,2018-06-30,53265000000,0000320193-19-000119,2019,FY,10-K,2019-10-31,CY2018Q2
9,2017-10-01,2018-09-29,265595000000,0000320193-19-000119,2019,FY,10-K,2019-10-31,NaN


In [72]:
annual_revenue["fp"].value_counts()

fp
FY    37
Name: count, dtype: int64

In [73]:
annual_revenue = annual_revenue[
    annual_revenue["fp"] == "FY"
].copy()

In [74]:
annual_revenue[["start", "end", "val", "fy", "fp", "form", "filed"]]

,start,end,val,fy,fp,form,filed
0,2016-09-25,2017-09-30,229234000000,2019,FY,10-K,2019-10-31
2,2017-10-01,2017-12-30,88293000000,2019,FY,10-K,2019-10-31
5,2017-12-31,2018-03-31,61137000000,2019,FY,10-K,2019-10-31
8,2018-04-01,2018-06-30,53265000000,2019,FY,10-K,2019-10-31
9,2017-10-01,2018-09-29,265595000000,2019,FY,10-K,2019-10-31
10,2017-10-01,2018-09-29,265595000000,2020,FY,10-K,2020-10-30
11,2018-07-01,2018-09-29,62900000000,2019,FY,10-K,2019-10-31
13,2018-09-30,2018-12-29,84310000000,2019,FY,10-K,2019-10-31
15,2018-09-30,2018-12-29,84310000000,2020,FY,10-K,2020-10-30
19,2018-12-30,2019-03-30,58015000000,2019,FY,10-K,2019-10-31


In [75]:
annual_revenue = annual_revenue.sort_values("fy")

In [76]:
annual_revenue[
    annual_revenue["fy"] >= 2020
][
    ["fy", "start", "end", "val", "form", "filed", "accn"]
].sort_values("fy")

,fy,start,end,val,form,filed,accn
10,2020,2017-10-01,2018-09-29,265595000000,10-K,2020-10-30,0000320193-20-000096
21,2020,2018-12-30,2019-03-30,58015000000,10-K,2020-10-30,0000320193-20-000096
27,2020,2019-03-31,2019-06-29,53809000000,10-K,2020-10-30,0000320193-20-000096
29,2020,2018-09-30,2019-09-28,260174000000,10-K,2020-10-30,0000320193-20-000096
15,2020,2018-09-30,2018-12-29,84310000000,10-K,2020-10-30,0000320193-20-000096
44,2020,2020-03-29,2020-06-27,59685000000,10-K,2020-10-30,0000320193-20-000096
32,2020,2019-06-30,2019-09-28,64040000000,10-K,2020-10-30,0000320193-20-000096
34,2020,2019-09-29,2019-12-28,91819000000,10-K,2020-10-30,0000320193-20-000096
39,2020,2019-12-29,2020-03-28,58313000000,10-K,2020-10-30,0000320193-20-000096
49,2020,2020-06-28,2020-09-26,64698000000,10-K,2020-10-30,0000320193-20-000096


In [77]:
annual_revenue["start"] = pd.to_datetime(annual_revenue["start"])
annual_revenue["end"] = pd.to_datetime(annual_revenue["end"])

annual_revenue["duration_days"] = (
    annual_revenue["end"] - annual_revenue["start"]
).dt.days

In [78]:
annual_revenue[
    ["fy", "start", "end", "duration_days", "val"]
].sort_values("end")

,fy,start,end,duration_days,val
0,2019,2016-09-25,2017-09-30,370,229234000000
2,2019,2017-10-01,2017-12-30,90,88293000000
5,2019,2017-12-31,2018-03-31,90,61137000000
8,2019,2018-04-01,2018-06-30,90,53265000000
9,2019,2017-10-01,2018-09-29,363,265595000000
11,2019,2018-07-01,2018-09-29,90,62900000000
10,2020,2017-10-01,2018-09-29,363,265595000000
13,2019,2018-09-30,2018-12-29,90,84310000000
15,2020,2018-09-30,2018-12-29,90,84310000000
19,2019,2018-12-30,2019-03-30,90,58015000000


In [79]:
annual_revenue_clean = annual_revenue[
    annual_revenue["duration_days"].between(350, 380)
].copy()

In [80]:
annual_revenue_clean[
    ["fy", "start", "end", "duration_days", "val", "filed"]
].sort_values("end")

,fy,start,end,duration_days,val,filed
0,2019,2016-09-25,2017-09-30,370,229234000000,2019-10-31
9,2019,2017-10-01,2018-09-29,363,265595000000,2019-10-31
10,2020,2017-10-01,2018-09-29,363,265595000000,2020-10-30
28,2019,2018-09-30,2019-09-28,363,260174000000,2019-10-31
29,2020,2018-09-30,2019-09-28,363,260174000000,2020-10-30
30,2021,2018-09-30,2019-09-28,363,260174000000,2021-10-29
46,2020,2019-09-29,2020-09-26,363,274515000000,2020-10-30
47,2021,2019-09-29,2020-09-26,363,274515000000,2021-10-29
48,2022,2019-09-29,2020-09-26,363,274515000000,2022-10-28
62,2023,2020-09-27,2021-09-25,363,365817000000,2023-11-03


In [81]:
annual_revenue_clean = annual_revenue_clean.sort_values(
    "filed"
)
annual_revenue_unique = annual_revenue_clean.drop_duplicates(
    subset=["start", "end"],
    keep="last"
).copy()

In [86]:
annual_revenue_unique["revenue_billions"] = (
    annual_revenue_unique["val"] / 1_000_000_000
)

annual_revenue_unique = annual_revenue_unique.sort_values("end")

In [87]:
annual_revenue_unique["fiscal_year"] = (
    annual_revenue_unique["end"].dt.year
)

In [88]:
annual_revenue_unique[
    [
        "fiscal_year",
        "start",
        "end",
        "revenue_billions",
        "filed"
    ]
]

,fiscal_year,start,end,revenue_billions,filed
0,2017,2016-09-25,2017-09-30,229.234,2019-10-31
10,2018,2017-10-01,2018-09-29,265.595,2020-10-30
30,2019,2018-09-30,2019-09-28,260.174,2021-10-29
48,2020,2019-09-29,2020-09-26,274.515,2022-10-28
62,2021,2020-09-27,2021-09-25,365.817,2023-11-03
75,2022,2021-09-26,2022-09-24,394.328,2024-11-01
88,2023,2022-09-25,2023-09-30,383.285,2025-10-31
100,2024,2023-10-01,2024-09-28,391.035,2025-10-31
111,2025,2024-09-29,2025-09-27,416.161,2025-10-31


In [89]:
annual_revenue_unique = annual_revenue_unique.sort_values(
    "fiscal_year"
).copy()
annual_revenue_unique["revenue_growth_pct"] = (
    annual_revenue_unique["revenue_billions"].pct_change() * 100
)

In [91]:
annual_revenue_unique[
    [
        "fiscal_year",
        "revenue_billions",
        "revenue_growth_pct"
    ]
]

,fiscal_year,revenue_billions,revenue_growth_pct
0,2017,229.234,NaN
10,2018,265.595,15.861958
30,2019,260.174,-2.041078
48,2020,274.515,5.512080
62,2021,365.817,33.259385
75,2022,394.328,7.793788
88,2023,383.285,-2.800461
100,2024,391.035,2.021994
111,2025,416.161,6.425512


In [92]:
final_revenue = annual_revenue_unique[
    [
        "fiscal_year",
        "start",
        "end",
        "revenue_billions",
        "revenue_growth_pct",
        "filed"
    ]
].copy()

In [93]:
final_revenue

,fiscal_year,start,end,revenue_billions,revenue_growth_pct,filed
0,2017,2016-09-25,2017-09-30,229.234,NaN,2019-10-31
10,2018,2017-10-01,2018-09-29,265.595,15.861958,2020-10-30
30,2019,2018-09-30,2019-09-28,260.174,-2.041078,2021-10-29
48,2020,2019-09-29,2020-09-26,274.515,5.512080,2022-10-28
62,2021,2020-09-27,2021-09-25,365.817,33.259385,2023-11-03
75,2022,2021-09-26,2022-09-24,394.328,7.793788,2024-11-01
88,2023,2022-09-25,2023-09-30,383.285,-2.800461,2025-10-31
100,2024,2023-10-01,2024-09-28,391.035,2.021994,2025-10-31
111,2025,2024-09-29,2025-09-27,416.161,6.425512,2025-10-31
